In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from keras.datasets import reuters
from keras.utils import pad_sequences

In [40]:
num_words=10_000
max_len=120

In [41]:
from pathlib import Path

In [42]:
dataset_path=Path.cwd()/"reuters_dataset"

In [43]:
import os
os.environ["KERAS_HOME"] = "/content/reuters_dataset"

In [44]:
(x_train,y_train),(x_test,y_test)=reuters.load_data(num_words=10_000)

In [45]:
print(x_train.shape, x_test.shape)

(8982,) (2246,)


In [46]:
x_train=pad_sequences(x_train,maxlen=max_len)
x_test=pad_sequences(x_test,maxlen=max_len)

In [47]:
x_train=torch.tensor(x_train,dtype=torch.long)
y_train=torch.tensor(y_train,dtype=torch.long)
x_test=torch.tensor(x_test,dtype=torch.long)
y_test=torch.tensor(y_test,dtype=torch.long)

In [48]:
train_dataset=TensorDataset(x_train,y_train)
test_dataset=TensorDataset(x_test,y_test)
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=64)
test_loader=DataLoader(test_dataset,shuffle=False,batch_size=32)


In [54]:
from torch.storage import HAS_NUMPY
class BiLSTM(nn.Module):
  def __init__(self,vocab_size=10000,embed_dim=128,hidden_dim=64,num_classes=46):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim,padding_idx=0)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True,bidirectional=True)
    self.fc = nn.Linear(hidden_dim * 2, num_classes)
  def forward(self,x):
    embed=self.embedding(x)
    _,(h_n,_)=self.lstm(embed)
    h=torch.cat((h_n[-2],h_n[-1]),dim=1)
    return self.fc(h)

In [55]:
model=BiLSTM(vocab_size=num_words,num_classes=46)


In [56]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [57]:
epochs=2

In [59]:
for epoch in range(epochs):
  model.train()
  total_loss,correct,total=0,0,0
  for x_b,y_b in train_loader:
    optimizer.zero_grad()
    output=model(x_b)
    loss=criterion(output,y_b)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    correct+=(output.argmax(dim=1)==y_b).sum().item()
    total+=y_b.size(0)
  print(f"Epoch: {epoch+1/epochs} | Loss: {total_loss} | Correct: {correct} | Total: {total}")



Epoch: 0.5 | Loss: 330.900962471962 | Correct: 3736 | Total: 8982
Epoch: 1.5 | Loss: 267.06256425380707 | Correct: 4637 | Total: 8982


In [60]:
model.eval()

BiLSTM(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (lstm): LSTM(128, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=46, bias=True)
)

In [66]:
with torch.no_grad():
  pred=model(x_test[0].unsqueeze(0))

correct=(pred.argmax(dim=1)==y_test).sum().item()
total=y_test.size(0)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f"Correct: {correct} | Incorrect: {total - correct} | Accuracy: {100.0*correct/total:.2f}%")

Correct: 1166 | Incorrect: 1080 | Accuracy: 51.91%
